# Agenda 

The goal of this notebook is to ensure eveything is setup for the workshop

## Setup

In [1]:
%pip install -r ../requirements.txt

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Imports

In [2]:
import dotenv
import os
import requests
import rich


In [3]:
dotenv.load_dotenv("../env_workshop")

True

In [4]:
using_own_keys_flag = os.environ.get("USING_OWN_KEYS") == "true"

In [5]:
if using_own_keys_flag:
    print ("will try to use user provided keys")

    assert os.environ.get("OPENAI_API_KEY") is not None, "Please set OPENAI_API_KEY in ../env_workshop"
    #assert os.environ.get("ARIZE_API_KEY") is not None, "Please set ARIZE_API_KEY in ../env_workshop"
else:
    print ("will use proxy server, so no need for user provided keys")

will use proxy server, so no need for user provided keys


## OpenAI validation

In [6]:
from langchain_openai import ChatOpenAI


In [7]:
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL")


In [8]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url=OPENAI_BASE_URL,
    #temperature=0.2,
    #max_tokens=512,
)

llm.invoke("what is the weather in seattle")

AIMessage(content="I can't provide real-time weather updates, but you can easily find the current weather in Seattle by checking a weather website or using a weather app on your smartphone. Is there anything else you'd like to know about Seattle or its climate?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 14, 'total_tokens': 60, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CZIzIyYNvOvuyRrEwcrIQITcSZRqe', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--0debc779-2b9b-4616-b5c6-69107429509b-0', usage_metadata={'input_tokens': 14, 'output_tokens': 46, 'total_tokens': 60, 'input_token_details': {'audio': 0, 

langchain has a thin wrapper around the openai client

## Tavilly



[Tavily](https://www.tavily.com/) is a service provider that enables agents to access the web

In [9]:
TAVILY_BASE_URL = os.environ.get("TAVILY_BASE_URL")
TAVILY_BASE_URL

'https://llm-proxy.np-training.dev/v1/tavily/search'

In [10]:
def tavily_search(query, **kw):
    tavily_api_key = os.environ.get("TAVILY_API_KEY")
    if tavily_api_key is None:
        r = requests.post(TAVILY_BASE_URL,
                        json={"query": query, **kw}, timeout=60)
        r.raise_for_status()
        return r.json()
    else:
        from tavily import TavilyClient
        tavily_client = TavilyClient(api_key=tavily_api_key)

        return tavily_client.search(
            query=query,
            **kw
        )


res = tavily_search("what is the weather in seattle")
print(res)

{'query': 'what is the weather in seattle', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'title': 'Weather in Seattle', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Seattle', 'region': 'Washington', 'country': 'United States of America', 'lat': 47.6064, 'lon': -122.3308, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1762530666, 'localtime': '2025-11-07 07:51'}, 'current': {'last_updated_epoch': 1762530300, 'last_updated': '2025-11-07 07:45', 'temp_c': 8.9, 'temp_f': 48.0, 'is_day': 1, 'condition': {'text': 'Partly cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/day/116.png', 'code': 1003}, 'wind_mph': 10.5, 'wind_kph': 16.9, 'wind_degree': 193, 'wind_dir': 'SSW', 'pressure_mb': 1019.0, 'pressure_in': 30.1, 'precip_mm': 1.58, 'precip_in': 0.06, 'humidity': 80, 'cloud': 50, 'feelslike_c': 6.3, 'feelslike_f': 43.4, 'windchill_c': 4.4, 'windchill_f': 39.9, 'heatindex_c': 7.7, 'heatindex_f': 45.9, 'dewpoint_c': 5.2, 'dewpoint_

In [11]:
rich.print(res)

{
    'query': 'what is the weather in seattle',
    'follow_up_questions': None,
    'answer': None,
    'images': [],
    'results': [
        {
            'title': 'Weather in Seattle',
            'url': 'https://www.weatherapi.com/',
            'content': "{'location': {'name': 'Seattle', 'region': 'Washington', 'country': 'United States of 
America', 'lat': 47.6064, 'lon': -122.3308, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1762530666, 
'localtime': '2025-11-07 07:51'}, 'current': {'last_updated_epoch': 1762530300, 'last_updated': '2025-11-07 07:45',
'temp_c': 8.9, 'temp_f': 48.0, 'is_day': 1, 'condition': {'text': 'Partly cloudy', 'icon': 
'//cdn.weatherapi.com/weather/64x64/day/116.png', 'code': 1003}, 'wind_mph': 10.5, 'wind_kph': 16.9, 'wind_degree':
193, 'wind_dir': 'SSW', 'pressure_mb': 1019.0, 'pressure_in': 30.1, 'precip_mm': 1.58, 'precip_in': 0.06, 
'humidity': 80, 'cloud': 50, 'feelslike_c': 6.3, 'feelslike_f': 43.4, 'windchill_c': 4.4, 'windchill_f': 39.9, 
'heatindex_c': 7.7, 'heatindex_f': 45.9, 'dewpoint_c': 5.2, 'dewpoint_f': 41.4, 'vis_km': 16.0, 'vis_miles': 9.0, 
'uv': 0.0, 'gust_mph': 16.2, 'gust_kph': 26.0}}",
            'score': 0.9699354,
            'raw_content': None
        },
        {
            'url': 'https://world-weather.info/forecast/usa/seattle/november-2025/',
            'title': 'Weather in Seattle in November 2025 (Washington)',
            'content': "* Weather * Weather in Seattle # Weather in Seattle in November 2025 * +55° 7.2 mph S 29.7 
inHg75 %07:54 am05:51 pm +57° +48° 7.6 mph S 29.8 inHg73 %06:55 am04:49 pm +57° +48° 7.2 mph S 29.9 inHg81 %06:57 
am04:48 pm +57° +48° +57° +48° +55° +48° +57° +48° +57° +46° +54° +46° +55° +46° +54° +46° +54° +46° +54° +46° +54°
+46° +50° +43° +48° +41° +50° +43° +52° +43° +52° +45° +50° +43° +50° +43° +50° +43° +50° +41° +48° +41° +50° +41° 
+50° +43° +48° +39° +48° +41° +48° +39° +48° +39° Weather in Washington, D.C.**+66°** Olympia**+68°** 
Redmond**+72°** Lynnwood**+64°** Leavenworth**+79°** Kirkland**+70°** Kent**+72°** Issaquah**+72°** 
Ferndale**+66°** Everett**+64°** Edmonds**+63°** Bremerton**+66°** world's temperature today +70° Temperature 
units",
            'score': 0.90830135,
            'raw_content': None
        },
        {
            'url': 
'https://weatherspark.com/h/m/913/2025/7/Historical-Weather-in-July-2025-in-Seattle-Washington-United-States',
            'title': 'Seattle July 2025 Historical Weather Data (Washington, United States)',
            'content': 'July 2025 Weather History in Seattle Washington, United States ; Seattle Temperature 
History July 2025 · 45°F · 45°F ; Hourly Temperature in July 2025 in Seattle.',
            'score': 0.7211353,
            'raw_content': None
        },
        {
            'url': 'https://www.weather25.com/north-america/usa/washington/seattle?page=month&month=November',
            'title': 'Seattle weather in November 2025 - Weather25.com',
            'content': 'United States England Australia Canada °F °C 3. United States 6. November Location was 
added to My Locations Location was removed from My Locations Click on a day for an hourly weather forecast The 
weather in Seattle in November is very cold with **temperatures between 4°C and 10°C**, warm clothes are a must. 
Temperatures Sun Hours November Click on a day for an hourly weather forecast November12 | Month | Temperatures | 
Rainy Days | Dry Days | Snowy Days | Rainfall | Weather | More details | Rain in Seattle in November ## Recommended
Hotels in Seattle We know that finding the ideal hotel in Seattle can be a hard task... Book NowFour Seasons Hotel 
Seattle What is the chance of rain in Seattle in November?',
            'score': 0.6922474,
            'raw_content': None
        },
        {
            'url': 'https://www.yahoo.com/news/23abc-evening-weather-july-11-001736032.html',
            'title': '23ABC Evening weather update July 11, 2025 - Yahoo',
            'content': 'The foreca

In [12]:
res = tavily_search("what is the best running shoes")
rich.print(res)

{
    'query': 'what is the best running shoes',
    'follow_up_questions': None,
    'answer': None,
    'images': [],
    'results': [
        {
            'url': 'https://www.runnersworld.com/gear/a19663621/best-running-shoes/',
            'title': "The 12 Best Running Shoes of 2025 - Runner's World",
            'content': '3. The 12 Best Running Shoes of 2025 # The 12 Best Running Shoes of 2025 * Best Running 
Shoes ## Best Running Shoes What makes the Ghost a Best Running Shoe mainstay is its Goldilocks comfort, especially
for new runners unsure of where to start when it comes to shoe shopping. “This is a solid stability shoe, and it 
provides adequate support and cushioning for longer runs,” said a tester. “I compare it most to the Nike Alphafly, 
not because of its weight or size but because of the very soft, marshmallowy feel of the cushioning underfoot,” 
said shoe tester Trevor Conde, who sports a 2:21 marathon PR. The 6 Best Nike Running ShoesSplit Shift: lululemon’s
First Lightweight Speed ShoeThe Fastest Shoes at the 2025 Chicago MarathonTested and Reviewed: Brooks Hyperion Max 
3',
            'score': 0.8559598,
            'raw_content': None
        },
        {
            'url': 'https://www.youtube.com/watch?v=MFpzCcf6Q9U',
            'title': 'The Best Running Shoe From Every Brand (100% honest review)',
            'content': 'The Best Running Shoe From Every Brand (100% honest review)\nBen Parkes\n303000 
subscribers\n8914 likes\n652719 views\n2 Jun 2025\nIf you enjoyed the video, please like, comment and subscribe! 
Thank you for watching!\n\nSave 10% site wide on training plans, hats, technical & casual apparel - (Code - 
YOUTUBE10) \nhttps://bit.ly/benparkesyoutube10\n\nCheck out:\nTraining Plans https://bit.ly/benparkesplans\nRunning
Hats https://bit.ly/benparkeshats\nTechnical Gear https://bit.ly/benparkestechgear\nFree Beginner Plans 
https://bit.ly/benparkesfreeplans\nShop our latest arrivals https://bit.ly/47MgROg\n\nFollow me here:\nStrava: 
https://www.strava.com/athletes/2310069\nInstagram: https://www.instagram.com/benparkes\nSecond channel 
https://www.youtube.com/@benparkestheextramile\n\n📱For support enquiries or business enquiries, please go to: 
https://www.benparkes.com and use the chat function. \n\n⏰ Timecodes ⏰\n0:00 Intro\n0:53 Nike\n2:49 Asics\n4:50 
New Balance\n6:59 Hoka\n9:22 Puma\n11:32 Saucony\n13:27 Brooks\n15:33 Mizuno\n17:28 On\n19:48 Adidas\n22:16 What 
would you pick?\n\nAbout this video: In today’s video I’m choosing what I think is the best shoe from every running
shoe brand here in 2025. Not necessarily the fastest or most expensive, but the ones which standout above all the 
others in the line up that offer something really unique and special. From race-day supershoes to high-mileage 
daily trainers and ultra-cushioned recovery shoes, this is your ultimate running shoe guide so whether you’re a 
beginner or training for your next marathon, there’s something in here for you!\n\nThis video is NOT sponsored. All
products mentioned have been bought 100% by Ben and the channel. Our mission here is to help runners of all 
abilities improve and enjoy running to their full potential. Showcasing the best running tips and advice, while 
reviewing the latest gear and taking your round some of the best races in the world!\n424 comments\n',
            'score': 0.850382,
            'raw_content': None
        },
        {
            'url': 'https://www.runnersworld.com/gear/a26028922/running-shoes-for-men/',
            'title': "The 10 Best Running Shoes for Men in 2025 - Runner's World",
            'content': '2. Running Shoes 3. The 10 Best Running Shoes for Men # The 10 Best Running Shoes for Men *
Best Running Shoes for Men New runners tend to ask me, “What’s the best running shoe?” My answer: it all depends on
the runner. ## Best Running Shoes for Men The Brooks Ghost has held a place on our best running shoes for a *long* 
time, and it’s my go-to recommendation for n

## LLM Monitoring with Arize Phoenix OTEL

It can be hard to monitor LLM calls especially when they are part of a larger workflow.

We will set up [Arize Phoenix](https://phoenix.arize.com/) OpenTelemetry to help with that.



In [13]:
PHOENIX_PROJECT_NAME=os.environ.get("PHOENIX_PROJECT_NAME")

In [14]:
if PHOENIX_PROJECT_NAME is None or PHOENIX_PROJECT_NAME in ("npatta01","anonymous","default",""):
    raise ValueError("Please set PHOENIX_PROJECT_NAME in ../env_workshop. Avoid using default project names to prevent conflicts.")

ValueError: Please set PHOENIX_PROJECT_NAME in ../env_workshop. Avoid using default project names to prevent conflicts.

In [15]:
from phoenix.otel import register
from opentelemetry import trace

from openinference.instrumentation.langchain import LangChainInstrumentor
if os.environ.get("PHOENIX_COLLECTOR_ENDPOINT"):
    # configure the Phoenix tracer
    tracer_provider = register(
        project_name=PHOENIX_PROJECT_NAME, 
        auto_instrument=False 
    )
    
else:
    tracer_provider = trace.NoOpTracerProvider()

LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
tracer = trace.get_tracer(__name__)


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: npatta01
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: llm-tracing.np-training.dev:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [16]:
from langchain.agents import create_agent

In [17]:
def get_weather(city: str) -> str:  
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="openai:gpt-4o-mini",   
    tools=[get_weather],  
    system_prompt="You are a helpful assistant"  
)

res = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in seattle"}]}
)

In [18]:
arize_project_response = requests.get(
    f"{os.environ.get('PHOENIX_COLLECTOR_ENDPOINT')}/v1/projects/{PHOENIX_PROJECT_NAME}",
)

data = arize_project_response.json()

print (data)
# View your spans at:
print (f"""
       
View your spans at:

https://llm-tracing.np-training.dev/projects/{data['data']['id']}/spans
       
       
       """)

{'data': {'name': 'npatta01', 'description': None, 'id': 'UHJvamVjdDo1'}}


View your spans at:

https://llm-tracing.np-training.dev/projects/UHJvamVjdDo1/spans


       


Trace of above call
![Trace of above call](../images/trace_setup_weather.png)


We will be building workflows where it will be difficult to monitor LLM calls.    

We will use the Arize to see actual execution.

## Closing Thoughts

If after the workshop, you want to use your own keys uncomment the section `# using own keys`

In [19]:
!cat ../env_workshop

USING_OWN_KEYS=false
PHOENIX_COLLECTOR_ENDPOINT="http://llm-tracing.np-training.dev"
OPENAI_BASE_URL="https://llm-proxy.np-training.dev/v1"
TAVILY_BASE_URL="https://llm-proxy.np-training.dev/v1/tavily/search"
OPENAI_API_KEY="..."
PHOENIX_PROJECT_NAME=npatta01
# replce PHOENIX_PROJECT_NAME with some id

# local setup
#OPENAI_BASE_URL="http://127.0.0.1:8080/v1"
#TAVILY_BASE_URL="http://127.0.0.1:8080/v1/tavily/search"

# using own keys
#USING_OWN_KEYS=true
#PHOENIX_COLLECTOR_ENDPOINT=""
#OPENAI_API_KEY="..."
#TAVILY_API_KEY=".."
#OPENAI_BASE_URL="https://api.openai.com/v1"
#TAVILY_BASE_URL="https://api.tavily.com/search"



## Conclusion
Your env should be setup properly now.